# Diffusion-LM on E2E NLG — Colab Pipeline
**CENG 467 Term Project — Zübeyr Almaho (300201023)**

Bu notebook Colab için güvenli ve resumable şekilde düzenlendi:
1. Her major stage sonunda artifact snapshot'ı Drive'a kaydedilir.
2. Her komut notebook çıktısına canlı akıtılır; süre ve exit code net biçimde yazılır.
3. Her çalışma için benzersiz bir `RUN_ID` oluşturulur.
4. Runtime düşerse setup hücrelerini yeniden çalıştırıp Drive snapshot'ından restore edebilirsin.

Notebook akışı:
1. GPU + Drive hazırlığı
2. Repo clone + dependencies
3. Runtime helper'ları ve autosave fonksiyonları
4. E2E NLG dataset cache
5. GPT-2 baseline eğitim / generation / eval
6. T5 baseline eğitim / generation / eval
7. Diffusion-LM eğitim / generation / eval
8. Sonuç karşılaştırma tablosu
9. Ablation çalıştırmaları
10. Sonuç paketleme + indirme
11. Drive snapshot'ından restore (opsiyonel)

**Runtime → Change runtime type → A100** en iyi seçenek. **L4** kabul edilebilir. **T4** çalışır ama diffusion daha yavaş olur.

**Önemli:** Save garantisi için Drive mount hücresini atlama. Bu notebook artık kalıcı artifact'ı Drive altında tutacak şekilde tasarlandı.

## 0. GPU kontrolü

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 0b. Drive mount + run klasörü

Bu hücre kalıcı save için zorunlu. Her stage snapshot'ı `MyDrive/diffusion-lm-ctg-runs/<RUN_ID>/` altına yazılır.

In [ ]:
from datetime import datetime
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

RUN_ID = datetime.now().strftime('%Y%m%d-%H%M%S')
DRIVE_RUNS_ROOT = Path('/content/drive/MyDrive/diffusion-lm-ctg-runs')
RUN_ROOT = DRIVE_RUNS_ROOT / RUN_ID
for subdir in ('status', 'artifacts', 'logs'):
    (RUN_ROOT / subdir).mkdir(parents=True, exist_ok=True)

print(f'RUN_ID: {RUN_ID}')
print(f'Drive snapshot root: {RUN_ROOT}')
print('Bu run altındaki tüm major stage artifact\'ları buraya kopyalanacak.')

## 1. Repo + dependencies

In [ ]:
import os
import subprocess
from pathlib import Path

repo_dir = Path('diffusion-lm-ctg')
if not repo_dir.is_dir():
    !git clone https://github.com/zubeyralmaho/diffusion-lm-ctg.git
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=False)

%cd diffusion-lm-ctg
!git rev-parse --short HEAD
!pip install -q -r requirements.txt

PROJECT_DIR = Path.cwd()
print(f'Project dir: {PROJECT_DIR}')
print(f'Current RUN_ID: {RUN_ID}')
print(f'Drive snapshot root: {RUN_ROOT}')

## 1b. Runtime helper'ları ve autosave

Bu yardımcı fonksiyonlar her komutu canlı loglar, exit code ve süre yazar, ardından istediğin artifact'ları Drive'a snapshot olarak kaydeder.

In [ ]:
import json
import shutil
import subprocess
import time
from datetime import datetime
from pathlib import Path

LOG_FILE = RUN_ROOT / 'logs' / 'runtime.log'
LAST_STATUS = None


def _append_log(message: str) -> None:
    with open(LOG_FILE, 'a', encoding='utf-8') as log_file:
        log_file.write(message)


def run_cmd(cmd: str, stage: str) -> dict:
    global LAST_STATUS
    print(f'\n===== {stage} =====')
    print(f'$ {cmd}')
    start = time.time()
    process = subprocess.Popen(
        cmd,
        shell=True,
        cwd=str(PROJECT_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        _append_log(line)
    exit_code = process.wait()
    elapsed = round(time.time() - start, 1)
    status = {
        'stage': stage,
        'command': cmd,
        'exit_code': exit_code,
        'elapsed_seconds': elapsed,
        'finished_at': datetime.now().isoformat(timespec='seconds'),
    }
    status_path = RUN_ROOT / 'status' / f'{stage}.json'
    status_path.write_text(json.dumps(status, indent=2), encoding='utf-8')
    LAST_STATUS = status
    print(f'[{stage}] exit_code={exit_code} elapsed={elapsed / 60:.2f} min')
    print(f'[{stage}] status saved to {status_path}')
    if exit_code != 0:
        raise RuntimeError(f'{stage} failed with exit code {exit_code}')
    return status


def save_stage(stage: str, relative_paths: list[str]) -> None:
    stage_dir = RUN_ROOT / 'artifacts' / stage
    if stage_dir.exists():
        shutil.rmtree(stage_dir)
    stage_dir.mkdir(parents=True, exist_ok=True)

    copied = []
    missing = []
    for relative_path in relative_paths:
        src = PROJECT_DIR / relative_path
        if not src.exists():
            missing.append(relative_path)
            continue
        dst = stage_dir / relative_path
        dst.parent.mkdir(parents=True, exist_ok=True)
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
        copied.append(relative_path)

    manifest = {
        'stage': stage,
        'saved_at': datetime.now().isoformat(timespec='seconds'),
        'copied': copied,
        'missing': missing,
        'run_id': RUN_ID,
    }
    manifest_path = stage_dir / 'manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(f'[{stage}] artifact snapshot saved to {stage_dir}')
    if missing:
        print(f'[{stage}] missing paths: {missing}')


def show_text(relative_path: str) -> None:
    target = PROJECT_DIR / relative_path
    print(target.read_text(encoding='utf-8'))


def restore_stage(run_id: str, stage: str) -> None:
    snapshot_dir = DRIVE_RUNS_ROOT / run_id / 'artifacts' / stage
    if not snapshot_dir.exists():
        raise FileNotFoundError(f'Snapshot not found: {snapshot_dir}')
    for item in snapshot_dir.iterdir():
        if item.name == 'manifest.json':
            continue
        dst = PROJECT_DIR / item.name
        if item.is_dir():
            shutil.copytree(item, dst, dirs_exist_ok=True)
        else:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, dst)
    print(f'Restored stage {stage} from run {run_id}')


run_metadata = {
    'run_id': RUN_ID,
    'project_dir': str(PROJECT_DIR),
    'drive_root': str(RUN_ROOT),
    'created_at': datetime.now().isoformat(timespec='seconds'),
}
(RUN_ROOT / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')
print('Runtime helpers ready.')
print(f'Live log file: {LOG_FILE}')

## 2. Dataset cache

In [ ]:
run_cmd('bash scripts/prepare_data.sh', 'prepare_data')
save_stage('prepare_data', ['data/raw/e2e'])

## 3. GPT-2 baseline — eğitim + generation + eval

In [ ]:
run_cmd('python -m src.train --config configs/gpt2_baseline.yaml', 'gpt2_train')
save_stage('gpt2_train', ['checkpoints/gpt2_baseline'])

run_cmd(
    'python -m src.generate --config configs/gpt2_baseline.yaml --split test --out results/generations/gpt2_baseline.jsonl',
    'gpt2_generate',
)
save_stage('gpt2_generate', ['results/generations/gpt2_baseline.jsonl'])

run_cmd(
    'python -m src.evaluate --predictions results/generations/gpt2_baseline.jsonl --out results/metrics/gpt2_baseline.json',
    'gpt2_eval',
)
save_stage(
    'gpt2_eval',
    [
        'checkpoints/gpt2_baseline',
        'results/generations/gpt2_baseline.jsonl',
        'results/metrics/gpt2_baseline.json',
    ],
)
show_text('results/metrics/gpt2_baseline.json')

## 4. T5 baseline — eğitim + generation + eval

In [ ]:
run_cmd('python -m src.train --config configs/t5_baseline.yaml', 't5_train')
save_stage('t5_train', ['checkpoints/t5_baseline'])

run_cmd(
    'python -m src.generate --config configs/t5_baseline.yaml --split test --out results/generations/t5_baseline.jsonl',
    't5_generate',
)
save_stage('t5_generate', ['results/generations/t5_baseline.jsonl'])

run_cmd(
    'python -m src.evaluate --predictions results/generations/t5_baseline.jsonl --out results/metrics/t5_baseline.json',
    't5_eval',
)
save_stage(
    't5_eval',
    [
        'checkpoints/t5_baseline',
        'results/generations/t5_baseline.jsonl',
        'results/metrics/t5_baseline.json',
    ],
)
show_text('results/metrics/t5_baseline.json')

## 5. Diffusion-LM (proposed method) — eğitim + generation + eval

En uzun adım. A100'de ~45-60 dk, T4'te 2-3 saat.

Önemli: Diffusion tarafında sampler, target-mask ve embedding initialization düzeltildi. Bu yüzden eski diffusion checkpoint'leri artık geçerli kabul edilmemeli. Aşağıdaki hücre temiz başlayacak şekilde eski diffusion artifact'larını siler, sonra sıfırdan train/generate/eval yapar.

In [ ]:
run_cmd('rm -rf checkpoints/diffusion_lm results/generations/diffusion_lm.jsonl results/metrics/diffusion_lm.json', 'diffusion_reset')

run_cmd('python -m src.train --config configs/diffusion_lm.yaml', 'diffusion_train')
save_stage('diffusion_train', ['checkpoints/diffusion_lm'])

run_cmd(
    'python -m src.generate --config configs/diffusion_lm.yaml --split test --out results/generations/diffusion_lm.jsonl',
    'diffusion_generate',
)
save_stage('diffusion_generate', ['results/generations/diffusion_lm.jsonl'])

run_cmd(
    'python -m src.evaluate --predictions results/generations/diffusion_lm.jsonl --out results/metrics/diffusion_lm.json',
    'diffusion_eval',
)
save_stage(
    'diffusion_eval',
    [
        'checkpoints/diffusion_lm',
        'results/generations/diffusion_lm.jsonl',
        'results/metrics/diffusion_lm.json',
    ],
)
show_text('results/metrics/diffusion_lm.json')

## 6. Karşılaştırma tablosu

Üç modelin metriklerini tek tabloda birleştir.

In [ ]:
run_cmd('python -m src.compare_results --metrics_dir results/metrics --out results/summary.md', 'compare_results')
save_stage('compare_results', ['results/summary.md', 'results/metrics'])
from IPython.display import Markdown
Markdown(open('results/summary.md', encoding='utf-8').read())

## 7. Ablation studies (opsiyonel — uzun)

5 ablation: cosine schedule, DDIM 50/100 steps, prefix length 16/48. Sampling-only ablation'lar base checkpoint'i yeniden kullanır; retraining ablation'ları yeni model eğitir.

Zamanın yoksa bu hücreyi atla — Zübeyr ikinci oturumda yapar.

In [ ]:
run_cmd('bash scripts/run_ablations.sh', 'ablations')
save_stage(
    'ablations',
    [
        'results/generations/ablations',
        'results/metrics/ablations',
        'results/summary.md',
        'checkpoints',
    ],
)

## 8. Sonuçları paketle ve indir

Bu hücre `results.zip` üretir ve Colab'dan tarayıcına indirir. Bu zip'i Zübeyr'e gönder.

In [ ]:
import shutil

archive_path = shutil.make_archive('results', 'zip', 'results')
run_cmd('ls -lh results.zip', 'package_results')
save_stage('package_results', ['results.zip', 'results'])

from google.colab import files
files.download('results.zip')
print(f'results.zip hazır. Drive snapshot: {RUN_ROOT / "artifacts" / "package_results"}')

## 9. Drive snapshot'ından restore et (opsiyonel)

Runtime düştüyse veya yeni bir Colab oturumuna geçtiysen önce 0b, 1 ve 1b hücrelerini tekrar çalıştır.
Sonra aşağıdaki hücrede `RESTORE_RUN_ID` ve `RESTORE_STAGE` değerlerini doldurup çalıştır.

Örnek stage adları: `prepare_data`, `gpt2_train`, `gpt2_generate`, `gpt2_eval`, `t5_train`, `diffusion_train`, `compare_results`, `ablations`, `package_results`.

In [ ]:
RESTORE_RUN_ID = ''
RESTORE_STAGE = ''

if not RESTORE_RUN_ID or not RESTORE_STAGE:
    print('Restore etmek için RESTORE_RUN_ID ve RESTORE_STAGE doldur.')
    print(f'Mevcut run klasörleri: {sorted(p.name for p in DRIVE_RUNS_ROOT.iterdir() if p.is_dir())[-5:]}')
else:
    restore_stage(RESTORE_RUN_ID, RESTORE_STAGE)